# MBEC V11A — Post-process existing V10 outputs

This notebook performs the practical, no-retraining analyses requested for the MBEC manuscript:

1. maps the implemented phase-difference metric to **pd-wPLI** for reporting;
2. exports per-patient alarm results for RF-F1, RF-F7, XGB-F1, and XGB-F6;
3. reports macro and pooled alarm metrics with confidence intervals;
4. adds paired effect sizes and bootstrap confidence intervals for MCC;
5. expands cross-fitting quality-control summaries.

It does **not** recompute features or refit any classifier.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.stats import chi2, friedmanchisquare, rankdata, wilcoxon

RANDOM_SEED = 42
N_BOOTSTRAP = 20000
V10_ROOT_OVERRIDE = None

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
OUTPUT_ROOT = (Path('/kaggle/working/MBEC_V11A_POSTPROCESS')
               if Path('/kaggle/working').exists()
               else Path.cwd() / 'MBEC_V11A_POSTPROCESS')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DISPLAY_MODEL = {
    'RandomForest': 'RF',
    'XGBoost': 'XGB',
    'SVM_RBF': 'SVM-RBF',
}
DISPLAY_FEATURE = {
    'F1_Handcrafted': 'F1 Handcrafted',
    'F2_Graph_PLI': 'F2 PLI',
    'F3_Graph_wPLI': 'F3 pd-wPLI',
    'F4_Graph_PLI_wPLI': 'F4 PLI+pd-wPLI',
    'F5_Handcrafted_PLI': 'F5 HC+PLI',
    'F6_Handcrafted_wPLI': 'F6 HC+pd-wPLI',
    'F7_All': 'F7 HC+PLI+pd-wPLI',
}
SELECTED_PAIRS = [
    ('RandomForest', 'F1_Handcrafted'),
    ('RandomForest', 'F7_All'),
    ('XGBoost', 'F1_Handcrafted'),
    ('XGBoost', 'F6_Handcrafted_wPLI'),
]

def discover_v10_root():
    if V10_ROOT_OVERRIDE:
        candidates = [Path(V10_ROOT_OVERRIDE)]
    else:
        candidates = []
        for base in SEARCH_ROOTS:
            if base.exists():
                candidates.extend(base.rglob('LOSO_MULTIMODEL_GAPAWARE_V10'))
    required = {
        'subject_metrics.csv', 'cohort_pooled_summary.csv',
        'alarm_sensitivity_subject_metrics.csv', 'crossfit_stream_qc.csv',
        'friedman_mcc_models.csv', 'run_config.json'
    }
    valid = []
    for root in candidates:
        if root.is_dir() and required.issubset({p.name for p in root.iterdir()}):
            valid.append(root.resolve())
    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError('Could not find LOSO_MULTIMODEL_GAPAWARE_V10. Set V10_ROOT_OVERRIDE.')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple V10 roots found: {valid}. Set V10_ROOT_OVERRIDE explicitly.')
    return valid[0]

V10_ROOT = discover_v10_root()
print('V10 root:', V10_ROOT)
print('V11A output:', OUTPUT_ROOT)


## 1. Terminology mapping


In [ ]:
terminology = pd.DataFrame([
    {'InternalCode': 'Graph_wPLI', 'ManuscriptTerm': 'phase-difference weighted Phase Lag Index',
     'Abbreviation': 'pd-wPLI',
     'Definition': '|sum sin(delta_phi)| / (sum |sin(delta_phi)| + epsilon)'},
    {'InternalCode': 'F3_Graph_wPLI', 'ManuscriptTerm': 'F3 pd-wPLI graph',
     'Abbreviation': 'F3', 'Definition': '485 graph features'},
    {'InternalCode': 'F4_Graph_PLI_wPLI', 'ManuscriptTerm': 'F4 PLI+pd-wPLI',
     'Abbreviation': 'F4', 'Definition': '970 graph features'},
    {'InternalCode': 'F6_Handcrafted_wPLI', 'ManuscriptTerm': 'F6 handcrafted+pd-wPLI',
     'Abbreviation': 'F6', 'Definition': '1385 raw features'},
    {'InternalCode': 'F7_All', 'ManuscriptTerm': 'F7 handcrafted+PLI+pd-wPLI',
     'Abbreviation': 'F7', 'Definition': '1870 raw features'},
])
terminology.to_csv(OUTPUT_ROOT / 'terminology_mapping.csv', index=False)
display(terminology)


## 2. Statistical helpers


In [ ]:
def add_display_names(frame):
    out = frame.copy()
    out['ModelDisplay'] = out['Model'].map(DISPLAY_MODEL).fillna(out['Model'])
    out['FeatureDisplay'] = out['FeatureSet'].map(DISPLAY_FEATURE).fillna(out['FeatureSet'])
    out['Configuration'] = out['ModelDisplay'] + '-' + out['FeatureDisplay'].str.split().str[0]
    return out

def bootstrap_mean_ci(values, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        boot[i] = np.mean(rng.choice(values, size=len(values), replace=True))
    return tuple(np.quantile(boot, [0.025, 0.975]))

def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return np.nan, np.nan
    p = successes / total
    den = 1 + z*z/total
    center = (p + z*z/(2*total)) / den
    half = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / den
    return center-half, center+half

def poisson_rate_interval(events, exposure_h, alpha=0.05):
    if exposure_h <= 0:
        return np.nan, np.nan
    lo = 0.0 if events == 0 else 0.5 * chi2.ppf(alpha/2, 2*events) / exposure_h
    hi = 0.5 * chi2.ppf(1-alpha/2, 2*(events+1)) / exposure_h
    return lo, hi

def rank_biserial_from_differences(differences):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d) & (d != 0)]
    if len(d) == 0:
        return np.nan
    ranks = rankdata(np.abs(d), method='average')
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    return (w_pos - w_neg) / (w_pos + w_neg)

def paired_hodges_lehmann(differences):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return np.nan
    walsh = [(d[i] + d[j]) / 2 for i in range(len(d)) for j in range(i, len(d))]
    return float(np.median(walsh))

def bootstrap_median_difference_ci(differences, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        boot[i] = np.median(rng.choice(d, size=len(d), replace=True))
    return tuple(np.quantile(boot, [0.025, 0.975]))


## 3. Selected per-patient alarm results


In [ ]:
subject_alarm = pd.read_csv(V10_ROOT / 'alarm_sensitivity_subject_metrics.csv')
primary = subject_alarm.loc[subject_alarm['IsPrimary'].astype(bool)].copy()
mask = pd.Series(False, index=primary.index)
for model, feature in SELECTED_PAIRS:
    mask |= (primary['Model'].eq(model) & primary['FeatureSet'].eq(feature))
selected_patient = add_display_names(primary.loc[mask])
selected_patient = selected_patient[[
    'Subject', 'Configuration', 'Model', 'FeatureSet', 'Seizures',
    'SeizuresPredicted', 'SeizureSensitivity', 'FalseAlarms',
    'EligibleInterictalHours', 'FAR_per_h', 'MeanTimeToWarning_min'
]].sort_values(['Configuration', 'Subject'])
selected_patient.to_csv(OUTPUT_ROOT / 'selected_primary_per_patient.csv', index=False)
display(selected_patient)


## 4. Macro and pooled event metrics


In [ ]:
pooled = pd.read_csv(V10_ROOT / 'cohort_pooled_summary.csv')
rows = []
for model, feature in SELECTED_PAIRS:
    patients = selected_patient.loc[
        selected_patient['Model'].eq(model) & selected_patient['FeatureSet'].eq(feature)]
    pool = pooled.loc[pooled['Model'].eq(model) & pooled['FeatureSet'].eq(feature)].iloc[0]
    ss_ci = wilson_interval(int(pool['SeizuresPredicted']), int(pool['Seizures']))
    far_ci = poisson_rate_interval(int(pool['FalseAlarms']), float(pool['EligibleInterictalHours']))
    macro_ss_ci = bootstrap_mean_ci(patients['SeizureSensitivity'], seed=RANDOM_SEED)
    macro_far_ci = bootstrap_mean_ci(patients['FAR_per_h'], seed=RANDOM_SEED + 1)
    macro_ttw_ci = bootstrap_mean_ci(patients['MeanTimeToWarning_min'], seed=RANDOM_SEED + 2)
    rows.append({
        'Configuration': f'{DISPLAY_MODEL[model]}-{feature.split("_")[0]}',
        'Model': model, 'FeatureSet': feature,
        'Subjects': len(patients), 'Seizures': int(pool['Seizures']),
        'PooledSeizuresPredicted': int(pool['SeizuresPredicted']),
        'PooledSeizureSensitivity': pool['SeizureSensitivity'],
        'PooledSS_CI95_Low': ss_ci[0], 'PooledSS_CI95_High': ss_ci[1],
        'PooledFalseAlarms': int(pool['FalseAlarms']),
        'PooledEligibleHours': pool['EligibleInterictalHours'],
        'PooledFAR_per_h': pool['FAR_per_h'],
        'PooledFAR_CI95_Low': far_ci[0], 'PooledFAR_CI95_High': far_ci[1],
        'PooledMeanTTW_min': pool['MeanTimeToWarning_min'],
        'MacroMeanSS': patients['SeizureSensitivity'].mean(),
        'MacroSS_CI95_Low': macro_ss_ci[0], 'MacroSS_CI95_High': macro_ss_ci[1],
        'MacroMeanFAR_per_h': patients['FAR_per_h'].mean(),
        'MacroFAR_CI95_Low': macro_far_ci[0], 'MacroFAR_CI95_High': macro_far_ci[1],
        'MedianPatientFAR_per_h': patients['FAR_per_h'].median(),
        'IQRPatientFAR_per_h': patients['FAR_per_h'].quantile(.75)-patients['FAR_per_h'].quantile(.25),
        'MacroMeanTTW_min': patients['MeanTimeToWarning_min'].mean(),
        'MacroTTW_CI95_Low': macro_ttw_ci[0], 'MacroTTW_CI95_High': macro_ttw_ci[1],
    })
macro_pooled = pd.DataFrame(rows)
macro_pooled.to_csv(OUTPUT_ROOT / 'selected_primary_macro_pooled.csv', index=False)
display(macro_pooled)


## 5. Effect sizes and uncertainty


In [ ]:
subject_metrics = pd.read_csv(V10_ROOT / 'subject_metrics.csv')
effect_rows = []
for model, model_df in subject_metrics.groupby('Model'):
    pivot = model_df.pivot(index='Subject', columns='FeatureSet', values='MCC')
    if 'F1_Handcrafted' not in pivot:
        continue
    for feature in sorted(c for c in pivot.columns if c != 'F1_Handcrafted'):
        pair = pivot[['F1_Handcrafted', feature]].dropna()
        d = pair[feature] - pair['F1_Handcrafted']
        stat, p = wilcoxon(d, alternative='two-sided', zero_method='wilcox')
        ci = bootstrap_median_difference_ci(d, seed=RANDOM_SEED)
        effect_rows.append({
            'Model': model,
            'Comparison': f'{DISPLAY_FEATURE.get(feature, feature)} vs F1 Handcrafted',
            'FeatureSet': feature,
            'N_PairedSubjects': len(d),
            'MedianDeltaMCC': d.median(),
            'MeanDeltaMCC': d.mean(),
            'HodgesLehmannDeltaMCC': paired_hodges_lehmann(d),
            'MedianDelta_CI95_Low': ci[0],
            'MedianDelta_CI95_High': ci[1],
            'RankBiserialCorrelation': rank_biserial_from_differences(d),
            'WilcoxonStatistic': stat,
            'WilcoxonP_Uncorrected': p,
        })
effects = pd.DataFrame(effect_rows)
effects['WilcoxonP_Holm'] = np.nan
for model, idx in effects.groupby('Model').groups.items():
    order = effects.loc[idx, 'WilcoxonP_Uncorrected'].sort_values().index
    m = len(order)
    adjusted = []
    running = 0.0
    for rank, row_idx in enumerate(order, start=1):
        value = min(1.0, effects.loc[row_idx, 'WilcoxonP_Uncorrected'] * (m-rank+1))
        running = max(running, value)
        adjusted.append((row_idx, running))
    for row_idx, value in adjusted:
        effects.loc[row_idx, 'WilcoxonP_Holm'] = value
effects.to_csv(OUTPUT_ROOT / 'mcc_effect_sizes_vs_F1.csv', index=False)
display(effects)

friedman_rows = []
for feature, group in subject_metrics.groupby('FeatureSet'):
    pivot = group.pivot(index='Subject', columns='Model', values='MCC').dropna()
    if len(pivot) and pivot.shape[1] >= 3:
        stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
        k = pivot.shape[1]
        kendall_w = stat / (len(pivot) * (k-1))
        friedman_rows.append({'FeatureSet': feature, 'N_Subjects': len(pivot),
                              'Models': k, 'FriedmanStatistic': stat,
                              'PValue': p, 'KendallW': kendall_w})
friedman_effects = pd.DataFrame(friedman_rows)
friedman_effects.to_csv(OUTPUT_ROOT / 'friedman_effect_sizes.csv', index=False)
display(friedman_effects)


## 6. Cross-fitting QC


In [ ]:
qc = pd.read_csv(V10_ROOT / 'crossfit_stream_qc.csv')
qc['CrossFitUnmatchedFraction'] = np.where(
    qc['EligibleInterictalEpochs'] > 0,
    qc['CrossFitUnmatchedInterictalEpochs'] / qc['EligibleInterictalEpochs'],
    np.nan)
qc['HasEligibleInterictal'] = (qc['EligibleInterictalEpochs'] > 0).astype(int)
qc['PredictionPolicy'] = 'matched epoch predicted by model holding out its owner seizure'
qc.to_csv(OUTPUT_ROOT / 'crossfit_qc_enhanced.csv', index=False)

summary = pd.DataFrame([{
    'OuterFolds': len(qc),
    'FoldsWithEligibleInterictal': int(qc['HasEligibleInterictal'].sum()),
    'FoldsWithoutEligibleInterictal': int((1-qc['HasEligibleInterictal']).sum()),
    'StreamEpochs': int(qc['StreamEpochs'].sum()),
    'EligibleInterictalEpochs': int(qc['EligibleInterictalEpochs'].sum()),
    'CrossFitMatchedInterictalEpochs': int(qc['CrossFitMatchedInterictalEpochs'].sum()),
    'CrossFitUnmatchedInterictalEpochs': int(qc['CrossFitUnmatchedInterictalEpochs'].sum()),
    'OverallCrossFitMatchFraction': (
        qc['CrossFitMatchedInterictalEpochs'].sum() / qc['EligibleInterictalEpochs'].sum()),
    'EligibleInterictalHours': qc['EligibleInterictalHours'].sum(),
}])
summary.to_csv(OUTPUT_ROOT / 'crossfit_qc_summary.csv', index=False)
display(summary)

checks = {
    'subjects': int(subject_metrics['Subject'].nunique()),
    'seizures': int(pd.read_csv(V10_ROOT/'fold_metrics.csv')[['Subject','HeldOutSeizure']].drop_duplicates().shape[0]),
    'selected_patient_rows': int(len(selected_patient)),
    'effect_rows': int(len(effects)),
    'all_selected_pairs_have_10_subjects': bool(
        selected_patient.groupby(['Model','FeatureSet'])['Subject'].nunique().eq(10).all()),
    'total_eligible_hours': float(qc['EligibleInterictalHours'].sum()),
}
with open(OUTPUT_ROOT / 'V11A_QC_report.json', 'w', encoding='utf-8') as f:
    json.dump(checks, f, indent=2)
print(json.dumps(checks, indent=2))
print('V11A post-processing complete:', OUTPUT_ROOT)
